# 02. Признаки answer + description

Второй эксперимент: извлекаем статистические признаки из двух текстовых полей, `answer` и `description`, и сравниваем качество с baseline.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNet, Lasso, LinearRegression, Ridge
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

## Загрузка данных

In [ ]:
DATA_PATH = Path("../data.csv")
df = pd.read_csv(DATA_PATH)
df.head()

## Извлечение текстовых признаков

In [ ]:
VOWELS = set("аеёиоуыэюя")
RARE_LETTERS = set("фщъёцэ")

def extract_text_stats(text, prefix=""):
    if pd.isna(text):
        text = ""
    
    # Для ответа берем только первый вариант до |
    text = str(text).lower().strip()
    if "|" in text:
        text = text.split("|")[0].strip()
    
    compact = text.replace(" ", "")
    words = [word for word in text.split() if word]
    
    letters = [char for char in compact if char.isalpha()]
    cyrillic_letters = [char for char in letters if "а" <= char <= "я" or char == "ё"]
    latin_letters = [char for char in letters if "a" <= char <= "z"]
    
    len_chars = len(compact)
    vowel_count = sum(char in VOWELS for char in compact)
    consonant_count = sum(char.isalpha() and char not in VOWELS for char in compact)
    rare_count = sum(char in RARE_LETTERS for char in compact)
    unique_chars = len(set(compact))
    
    stats = {
        f"{prefix}len_chars": len_chars,
        f"{prefix}len_words": len(words),
        f"{prefix}avg_word_len": len_chars / len(words) if words else 0.0,
        f"{prefix}vowel_count": vowel_count,
        f"{prefix}consonant_count": consonant_count,
        f"{prefix}rare_count": rare_count,
        f"{prefix}unique_chars": unique_chars,
        f"{prefix}vowel_ratio": vowel_count / len_chars if len_chars else 0.0,
        f"{prefix}rare_ratio": rare_count / len_chars if len_chars else 0.0,
        f"{prefix}unique_ratio": unique_chars / len_chars if len_chars else 0.0,
        f"{prefix}cyrillic_ratio": len(cyrillic_letters) / len(letters) if letters else 0.0,
        f"{prefix}latin_ratio": len(latin_letters) / len(letters) if letters else 0.0,
        f"{prefix}has_dash": int("-" in text),
        f"{prefix}has_digits": int(any(char.isdigit() for char in text)),
    }
    return pd.Series(stats)

answer_features = df["answer"].apply(lambda x: extract_text_stats(x, prefix="ans_"))
desc_features = df["description"].apply(lambda x: extract_text_stats(x, prefix="desc_"))

X = pd.concat([answer_features, desc_features], axis=1)
y = df["difficulty"]

X.head()

## Обучение моделей

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape

In [ ]:
RANDOM_STATE = 42

def scaled_model(model):
    return Pipeline([("scaler", StandardScaler()), ("model", model)])

models = {
    "dummy_mean": DummyRegressor(strategy="mean"),
    "linear_regression": scaled_model(LinearRegression()),
    "ridge": scaled_model(Ridge(alpha=1.0)),
    "lasso": scaled_model(Lasso(alpha=0.001, max_iter=10000, random_state=RANDOM_STATE)),
    "elastic_net": scaled_model(ElasticNet(alpha=0.001, l1_ratio=0.5, max_iter=10000, random_state=RANDOM_STATE)),
    "svr_rbf": scaled_model(SVR(kernel="rbf", C=1.0, epsilon=0.05)),
    "knn_5": scaled_model(KNeighborsRegressor(n_neighbors=5, weights="distance")),
    "random_forest": RandomForestRegressor(n_estimators=500, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
    "extra_trees": ExtraTreesRegressor(n_estimators=500, max_depth=5, min_samples_leaf=5, random_state=RANDOM_STATE),
    "gradient_boosting": GradientBoostingRegressor(n_estimators=200, learning_rate=0.03, max_depth=2, min_samples_leaf=5, random_state=RANDOM_STATE),
}

def evaluate_holdout(models, X_train, X_test, y_train, y_test):
    rows = []
    fitted_models = {}
    for name, model in models.items():
        m = clone(model)
        m.fit(X_train, y_train)
        y_pred = m.predict(X_test)
        rows.append({"model": name, "r2": r2_score(y_test, y_pred), "mae": mean_absolute_error(y_test, y_pred)})
        fitted_models[name] = m
    return pd.DataFrame(rows).sort_values("mae").reset_index(drop=True), fitted_models

holdout_metrics, fitted_models = evaluate_holdout(models, X_train, X_test, y_train, y_test)
holdout_metrics

## Кросс-валидация

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring={"r2": "r2", "mae": "neg_mean_absolute_error"})
    cv_rows.append({"model": name, "r2_mean": scores["test_r2"].mean(), "mae_mean": -scores["test_mae"].mean()})

cv_metrics = pd.DataFrame(cv_rows).sort_values("mae_mean").reset_index(drop=True)
cv_metrics

## Лучшая модель и визуализация ошибок

In [ ]:
best_model_name = cv_metrics.iloc[0]["model"]
best_model = fitted_models[best_model_name]
y_pred = best_model.predict(X_test)

plot_df = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
}).sort_values("Actual").reset_index(drop=True)

plt.figure(figsize=(12, 6))
plt.plot(plot_df["Actual"].values, label="Actual", linewidth=2)
plt.plot(plot_df["Predicted"].values, label="Predicted", alpha=0.7)
plt.title(f"Actual vs Predicted Difficulty (Sorted by Actual, Model: {best_model_name})")
plt.xlabel("Sample Index (sorted by actual difficulty)")
plt.ylabel("Difficulty")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Важность признаков

In [ ]:
importance = permutation_importance(
    best_model, X_test, y_test, n_repeats=30, random_state=RANDOM_STATE, scoring="neg_mean_absolute_error"
)
feature_importance = pd.DataFrame({
    "feature": X.columns,
    "importance_mean": importance.importances_mean
}).sort_values("importance_mean", ascending=False)
feature_importance.head(20)